# KVQuant Implementation -- full-precision baseline only

One of four split-out notebooks (2-bit / 3-bit / 4-bit / full-precision
baseline), each running independently so they can execute in parallel
Colab sessions instead of one long combined run. This notebook loads and
evaluates ONLY the untouched, unquantized full-precision model -- no
KVCacheCompression repo clone, no patches, no Fisher calibration, no
Quantize step, since none of that machinery applies to a model that never
gets quantized. The three quantized bit-widths each live in their own
dedicated notebook (`KVQuant_2bit_Implementation.ipynb`,
`_3bit_`, `_4bit_`), so this baseline only needs to run once rather than
being redundantly repeated in all three.

Every fix already built into v3 is preserved as-is: teacher-forced
step-by-step TTFT/TBT timing for WikiText-103, real
`generate()`-based GSM8K timing with TBT excluding TTFT, H2O-matching deterministic random sampling (up to 256 non-overlapping
WikiText-103 chunks and up to 1,024 valid examples from GSM8K,
ARC-Challenge, and HellaSwag, all with seed 42), and
download-retry-with-cache-clear robustness for every network call. Memory
is measured directly from the real KV cache tensors this model actually
produces (no outlier-hook simulation needed, since nothing here is
quantized).

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [ ]:
!hostname

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods (kept
# identical to the 2-bit/3-bit/4-bit notebooks even though this one never
# clones/patches the KVCacheCompression repo, so there's no risk of a
# transformers/tokenizers/etc. version mismatch skewing the comparison).

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) for consistency with the KVQuant-family notebooks,
# where the Fisher calibration cell needs them available early in Setup.
# This notebook has no such dependency itself (no calibration step), but
# keeping the split identical across the whole notebook family avoids
# Helper Functions containing different things in different notebooks.
# sync_if_cuda/clear_memory have no early dependency anywhere and live in
# Helper Functions with the rest of the genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Shared sampling policy used by every H2O and KVQuant comparison notebook:
#   - WikiText-103: up to 256 random non-overlapping chunks
#   - GSM8K, ARC-Challenge, HellaSwag: up to 1,024 random valid examples
#   - deterministic seed: 42
# Sampling happens after chunk construction or validity filtering. Selected
# indices are sorted back into source order so the subset is random while the
# evaluation order remains stable across methods. No ABITS/SPARSITY_THRESHOLD/
# quantizer settings here -- this notebook never quantizes anything.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
C = 2048                    # chunk length for WikiText-103 = model's max context
WIKITEXT_EVAL_SAMPLES = 256
QA_EVAL_SAMPLES = 1024

# Backward-compatible aliases used by a few downstream cells/comments.
N_EVAL_SAMPLES = WIKITEXT_EVAL_SAMPLES
N_QA_SAMPLES = QA_EVAL_SAMPLES
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_baseline_full_precision"

random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)


def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)
    sample_count = min(int(max_samples), len(items))
    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )
    return [items[index] for index in selected_indices], selected_indices

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("Random sampling seed:", SHARED_SEED)
print("WikiText-103 random chunk target:", WIKITEXT_EVAL_SAMPLES)
print("GSM8K/ARC/HellaSwag random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)


In [ ]:
# Block - Load tokenizer + the untouched full-precision model. No repo
# clone, no patches, no Fisher calibration, no Quantize step -- none of
# that machinery is needed for a model that's never quantized. This is the
# exact same loading code the combined v3 notebook used for model_fp,
# unchanged, so the baseline is byte-for-byte comparable across notebooks.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading full-precision baseline model...")
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_fp = model_fp.to(DEVICE)
model_fp.eval()
model_fp.config.use_cache = True

device = next(model_fp.parameters()).device
print("model_fp: full-precision baseline, loaded and ready.")

## Helper Functions

Shared inference machinery used across all three datasets (WikiText-103,
GSM8K, ARC-Challenge).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - KV-cache memory measurement (measured from REAL cache tensors)
# -- shared inference machinery used by both the WikiText-103 driver (via
# score_chunk_kvquant, WikiText-103 section) and directly by
# ARC-Challenge's gold-choice scoring (ARC-Challenge section). GSM8K
# measures its own memory inline in generate_gsm8k_kvquant instead of
# calling this, but this is still genuine cross-dataset machinery, not
# something specific to any one dataset's driver logic.
#
# Always run as a SEPARATE, UNTIMED pass -- callers run it AFTER their own
# timed loop finishes, so it cannot affect any timing number.
#
# No outlier-hook tracker exists here (nothing is quantized), so memory is
# measured directly: an untimed forward pass with use_cache=True, then
# summing the real byte size of the real past_key_values tensors it
# produced.


@torch.no_grad()
def measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)

    if tracker is not None:
        reset_outlier_tracker(tracker)
        model_obj(input_ids=chunk_ids, use_cache=False, return_dict=True)
        return measure_bytes_from_tracker(tracker, bits_per_element)

    outputs = model_obj(input_ids=chunk_ids, use_cache=True, return_dict=True)
    pkv = outputs.past_key_values
    legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
    total_bytes = 0
    for layer_kv in legacy:
        for t in layer_kv:
            total_bytes += t.numel() * t.element_size()
    return total_bytes

In [ ]:
# Block - Shared multiple-choice (MC) scoring machinery, used by BOTH
# ARC-Challenge and HellaSwag (the two likelihood-based MC datasets in
# this notebook) -- genuinely dataset-agnostic: each dataset's own section
# only builds (prompt, choices, gold_index) and calls
# score_mc_question_kvquant; everything else lives here.
#
# lm_eval_encode_pair(context, choice)
#     Tokenizes context+continuation JOINTLY then splits the result at the
#     context's own token count (matching lm-evaluation-harness) --
#     preserves true tokenizer boundary behavior, which separately
#     tokenizing context and continuation can miss (BPE merges can span
#     the boundary). Left-truncates the context if the combined request
#     would exceed the model's window C.
#
# score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
#     Times the FULL step-by-step walk over one answer choice's tokens
#     (teacher-forced, exactly like scoring a WikiText-103 chunk) --
#     EVERY choice for a question gets this same real, timed treatment,
#     not just the correct one, since answering a multiple-choice
#     question genuinely requires scoring every candidate (there's no
#     single "real generation" the way GSM8K has). Memory is measured via
#     measure_chunk_kv_memory (previous cell) over the full processed
#     sequence.
#
# score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker)
#     Scores every choice via score_mc_choice_kvquant, then reports BOTH
#     standard MC accuracy metrics: raw (argmin of un-normalized NLL) and
#     normalized (argmin of NLL divided by the choice's own raw character
#     length -- the standard lm-evaluation-harness acc_norm convention).
#     Perplexity comes ONLY from the gold choice's own mean NLL (mirroring
#     GSM8K/ARC's "score only the gold answer" convention for perplexity).
#     TTFT = mean across choices; TBT = weighted mean across every
#     choice's decode steps (not a mean-of-per-choice-TBTs, since choices
#     can have very different lengths); total latency = SUM across
#     choices (the real cost of answering the question is the cost of
#     scoring every choice); peak memory = MAX across choices.


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")
    if len(continuation_ids) > int(C):
        raise ValueError(f"Choice has {len(continuation_ids)} tokens, exceeding C={C}.")

    max_context_tokens = int(C) + 1 - len(continuation_ids)
    if max_context_tokens < 1:
        raise ValueError("No room remains for a context token.")
    context_ids = context_ids[-max_context_tokens:]

    return context_ids, continuation_ids


@torch.no_grad()
def score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker=None):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, full_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker=None):
    choice_results = [
        score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
        for choice in choices
    ]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }

## WikiText-103

In [ ]:
# Block - WikiText-103 random sampling.
# Load the FULL test text, concatenate it into one token stream, split it into
# non-overlapping chunks of length C, then randomly select up to 256 chunks
# with seed 42. The random subset is identical across methods. Selected chunk
# indices are sorted into source order before evaluation so timing is not
# affected by an arbitrary shuffled execution order.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [text for text in testdata["text"] if len(text.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    all_chunks = chunk_sequence(ids, C)
    selected_chunks, selected_indices = seeded_subset(
        all_chunks,
        WIKITEXT_EVAL_SAMPLES,
        SHARED_SEED,
    )

    print(
        f"WikiText-103 test set: {len(texts)} non-empty lines, "
        f"{ids.shape[0]} tokens, {len(all_chunks)} available non-overlapping chunks"
    )
    print(
        f"WikiText-103 selected: {len(selected_chunks)} random chunks "
        f"(requested {WIKITEXT_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("WikiText-103 selected chunk indices (first 20):", selected_indices[:20])
    return selected_chunks


wikitext103_chunks = load_wikitext103_chunks()


In [ ]:
# Block - WikiText-103 step-by-step eval function. TTFT/TBT/perplexity/
# accuracy come ONLY from the timed step-by-step loop below -- the model
# is stepped one token at a time using its own KV cache, teacher-forced
# (real corpus tokens fed in, never the model's own prediction). Memory is
# measured via measure_chunk_kv_memory (Helper Functions, Setup) in a
# SEPARATE, UNTIMED pass run AFTER the timed loop finishes, so it cannot
# affect any timing number.


@torch.no_grad()
def score_chunk_kvquant(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def preview_chunk_prediction(model_obj, chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate forward pass over just the first n_preview_tokens of
    a chunk -- for printing what the model actually predicted vs. the real
    next tokens. Does not affect any reported metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    outputs = model_obj(input_ids=input_ids, use_cache=False, return_dict=True)
    preds = outputs.logits.argmax(dim=-1)

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds[0], skip_special_tokens=True),
    }

In [ ]:
# Block - WikiText-103 driver: run every chunk through
# score_chunk_kvquant against the untouched full-precision model.


def evaluate_lm_dataset_kvquant(dataset_name, chunks, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []
    per_chunk_records = []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(model_obj, chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_kvquant(model_obj, chunk_ids, bits_per_element, tracker)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        chunk_scored = result["scored"]
        per_chunk_records.append({
            "chunk_index": chunk_idx,
            "prompt": tokenizer.decode(chunk_ids, skip_special_tokens=True),
            "accuracy": (result["correct"] / chunk_scored) if chunk_scored > 0 else float("nan"),
            "perplexity": math.exp(min(result["nll_sum"] / chunk_scored, 50.0)) if chunk_scored > 0 else float("nan"),
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_wikitext103_per_prompt.csv"
    pd.DataFrame(per_chunk_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_chunk_records)} per-chunk {dataset_name} rows to {_per_prompt_path}")

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_kvquant(_name, _chunks, model_fp, 16, METHOD_NAME))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

## GSM8K

In [ ]:
# Block - GSM8K loading: canonical test split, then random sampling.
# Build the complete list of valid question/answer pairs first, then select up
# to 1,024 examples with seed 42. This guarantees every method sees the same
# reproducible random questions rather than the first examples in the split.


def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

all_gsm8k_pairs = []
for item in gsm8k_test:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)
print("GSM8K selected valid-item indices (first 20):", gsm8k_selected_indices[:20])


In [ ]:
# Block - GSM8K generation with the full-precision baseline. Uses the real
# model.generate() call -- prefill processes the whole prompt in one shot,
# then decode continues one token at a time exactly like a normal
# generate() loop. A StoppingCriteria timestamps every generated token so
# TTFT/TBT come from this SAME call that produces the graded answer -- not
# a separate probe. Memory is measured in a SEPARATE, untimed pass after
# generate() returns, so it cannot affect any timing number.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same call -- no separate teacher-forced pass. generate() is called
# with output_scores=True, return_dict_in_generate=True, which returns the
# per-step logits it already computes internally (no extra forward pass)
# alongside the token ids. After generate() returns (i.e. fully outside
# the timed window), we scan the generated tokens for four consecutive
# '#' characters (the "####" marker GSM8K answers use), then accumulate
# the log-probability of each token the model chose for itself, starting
# right after the marker and stopping the moment another '#' appears. If
# the model never generates "####" at all, perplexity is None for that
# question.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


class _TimingCriteria(StoppingCriteria):
    def __init__(self):
        self.token_times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.token_times.append(time.perf_counter())
        return False


@torch.no_grad()
def generate_gsm8k_kvquant(model_obj, question, bits_per_element, tracker=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    timing = _TimingCriteria()
    sync_if_cuda()
    gen_start = time.perf_counter()
    gen_out = model_obj.generate(
        **enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=StoppingCriteriaList([timing]),
        output_scores=True, return_dict_in_generate=True,
    )
    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_ids = gen_out.sequences
    step_scores = gen_out.scores

    gen_text = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = len(timing.token_times)
    if n_generated > 0:
        ttft_sec = timing.token_times[0] - gen_start
    else:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Perplexity of the model's own predicted final number -- everything
    # below runs after gen_end, so none of it can affect any timing number.
    # step_scores[i] are the logits that chose the i-th generated token
    # (already computed by generate() above -- no extra forward pass).
    answer_token_ids = gen_ids[0][prompt_len: prompt_len + len(step_scores)].tolist()
    hash_streak = 0
    in_answer_span = False
    nll_sum = 0.0
    scored = 0
    for i, tok_id in enumerate(answer_token_ids):
        tok_text = tokenizer.decode([tok_id])
        if in_answer_span:
            if "#" in tok_text:
                break
            log_probs = torch.log_softmax(step_scores[i][0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
            scored += 1
        else:
            for ch in tok_text:
                hash_streak = hash_streak + 1 if ch == "#" else 0
            if hash_streak >= 4:
                in_answer_span = True
    perplexity = math.exp(min(nll_sum / scored, 50.0)) if scored > 0 else None

    # Untimed pass over the full generated sequence, purely to measure
    # memory -- real cache tensor bytes for the full-precision baseline
    # (no outlier-hook tracker needed, nothing here is quantized). Does
    # not affect the timed generate() call above.
    if tracker is not None:
        reset_outlier_tracker(tracker)
        model_obj(input_ids=gen_ids, use_cache=False, return_dict=True)
        peak_bytes = measure_bytes_from_tracker(tracker, bits_per_element)
    else:
        cache_outputs = model_obj(input_ids=gen_ids, use_cache=True, return_dict=True)
        pkv = cache_outputs.past_key_values
        legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
        peak_bytes = sum(t.numel() * t.element_size() for layer_kv in legacy for t in layer_kv)

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_kvquant
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass) -- against the
# full-precision baseline.


def evaluate_gsm8k_kvquant(qa_pairs, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_kvquant(model_obj, qa["question"], bits_per_element, tracker)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "prompt": qa["question"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": ppl,
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_kvquant(gsm8k_qa_pairs, model_fp, 16, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

## ARC-Challenge

Likelihood-based multiple-choice scoring (no generation): every answer
choice is scored (raw and character-length-normalized), perplexity from
the gold choice, TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - ARC-Challenge loading: canonical test split, validity filtering,
# then random sampling. Invalid answer-key rows are removed before selecting
# up to 1,024 valid questions with seed 42.


def load_arc_challenge_items():
    testdata = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )

    valid_items = []
    for row in testdata:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available; "
        f"selected {len(selected_items)} random questions "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_kvquant (Helper Functions),
# then reports character-length normalized MC accuracy. Aggregation
# mirrors GSM8K: perplexity = MEAN of per-question perplexities (from
# each question's gold choice), TTFT/TBT/latency = means over questions,
# peak_memory_gb = max over questions, average_memory_gb = mean over
# questions.


def evaluate_arc_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_kvquant(model_obj, prompt, choice_texts, gold_index, bits_per_element, tracker)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["question"],
            "choices": item["choices"],
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_kvquant(arc_items, model_fp, 16, METHOD_NAME),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)

## HellaSwag

Likelihood-based multiple-choice scoring (no generation), same
methodology as ARC-Challenge: every answer choice is scored (raw and
character-length-normalized), perplexity from the gold ending,
TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - HellaSwag loading: canonical validation split, preprocessing,
# then random sampling. HellaSwag's test labels are private, so validation is
# the standard evaluation split. Up to 1,024 processed examples are selected
# with seed 42.


def _hellaswag_preprocess(text):
    """Match lm-evaluation-harness HellaSwag preprocessing."""
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def load_hellaswag_items():
    dataset = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )

    processed_items = []
    for row in dataset:
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = _hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [_hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} validation examples available; "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_kvquant (Helper Functions) -- the same
# machinery ARC-Challenge uses, since both are likelihood-based
# multiple-choice datasets. Aggregation is identical to ARC-Challenge:
# normalized accuracy, perplexity = MEAN of per-question perplexities
# (from each example's gold ending), TTFT/TBT/latency = means over
# examples, peak_memory_gb = max over examples, average_memory_gb =
# mean over examples.


def evaluate_hellaswag_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_kvquant(
            model_obj, item["prompt"], item["choices"], item["gold_index"], bits_per_element, tracker,
        )

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["prompt"],
            "choices": item["choices"],
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_kvquant(hellaswag_items, model_fp, 16, METHOD_NAME),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)

## Save Results

In [ ]:
# Block - Combine WikiText-103 / GSM8K / ARC-Challenge / HellaSwag results into
# one table with ONLY the specified metrics, then save to CSV. This notebook only ever
# produces "kvquant_baseline_full_precision" rows -- 2-bit, 3-bit, and
# 4-bit are saved by their own separate notebooks, so there is no
# cross-notebook filename collision risk.
#
# Robust to partial runs: only concatenates whichever of
# lm_results_df / gsm8k_results_df / arc_results_df / hellaswag_results_df
# actually exist in this session, so you can run just a subset of
# datasets' cells without this cell crashing on a NameError for a
# dataframe you never created.

_result_df_names = ["lm_results_df", "gsm8k_results_df", "arc_results_df", "hellaswag_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_baseline_full_precision_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")